# Chunk Documents in Documents/ and Embed afterwards

In this file, we chunk all documents in the Documents/ directory and save them as a LangChain Document file.

Afterwards, we use an embedder to save a vector representation of each chunk.

## Read in the files

We first need to read the files we want to save in our documents.

In [6]:
import os
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    DirectoryLoader,
)

# Pad naar je Documents directory
documents_dir = "Documents/"

# Lijst om alle geladen documenten in op te slaan
loaded_documents = []

# Loop door alle subdirectories en bestanden
for root, dirs, files in os.walk(documents_dir):
    for file in files:
        file_path = os.path.join(root, file)

        # Laad tekstbestanden
        if file.endswith(".txt"):
            loader = TextLoader(file_path)
            documents = loader.load()
            loaded_documents.extend(documents)

        # Laad PDF-bestanden
        elif file.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
            documents = loader.load()
            loaded_documents.extend(documents)

        # Voeg hier andere bestandstypen toe indien nodig (bijv. .docx)

print(f"Aantal geladen documenten: {len(loaded_documents)}")

Aantal geladen documenten: 52


## Chunk the files

Chunk the files so that we can perform search later.
For now we stick to simple RecursiveCharacterSplitter - splitting the documents into similarly sized chunks.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configureer de splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,  # Aantal tekens per chunk
    chunk_overlap=100,  # Overlap tussen chunks
    separators=["\n\n", "\n", ". "]  # Prioriteit voor paragrafen, zinnen
)

# Chunk alle documenten
chunks = text_splitter.split_documents(loaded_documents)

print(f"Aantal chunks: {len(chunks)}")


Aantal chunks: 5654


In [25]:
print(chunks[1].page_content)
print(chunks[1].metadata)


Bij de geactualiseerde Werkwijzer Poortwachter van 1 augustus 2022
In deze werkwijzer is in paragraaf 6.3 de bijzondere arbeidsverhouding uitgewerkt van arbeid met 
een WSW-indicatie en beschut werk vanuit de Participatiewet. Ook zijn op basis van vragen, 
opmerkingen en suggesties teksten aangepast en instructies verduidelijkt. 
Deze tekstwijzigingen zijn gemarkeerd met een blauwe streep in de linker kantlijn.
Addendum COVID-19
{'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Windows)', 'creationdate': '2022-08-04T16:16:40+02:00', 'moddate': '2022-08-11T15:31:23+02:00', 'trapped': '/False', 'source': 'Documents/WerkwijzerPoortwachter\\WerkwijzerPoortwachter.pdf', 'total_pages': 42, 'page': 1, 'page_label': '2'}
{'extra': 'ignore'}


### Save the chunks to python file

We want to save the chunks so that we know which chunks we are recovering.

In [26]:
import pickle

def save_to_pickle(obj, filename):
    with open(filename, "wb") as file:
        pickle.dump(obj, file, pickle.HIGHEST_PROTOCOL)

save_to_pickle(chunks, "chunks.pickle")

## Embedding the chunks

Now that we have the chunks, let's embed them.

In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss  #pip intall faiss-cpu
import os

#We load the intfloat/multilingual-e5-base embedding model.
embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

In [20]:
embeddings = embedding_model.encode([chunk.page_content for chunk in chunks], show_progress_bar=True)

Batches: 100%|██████████| 177/177 [03:59<00:00,  1.35s/it]


### Finally, we save the embeddings in a FAISS vector database

In [21]:
d = embeddings.shape[1]
BeanRAG_VectorDB = faiss.IndexFlatL2(d)
BeanRAG_VectorDB.add(embeddings)
print("Total number of embeddings", BeanRAG_VectorDB.ntotal)
faiss.write_index(BeanRAG_VectorDB, "BeanRAG_VectorDB.faiss")
#To read the vector database, use faiss.read_index("BeanRAG_VectorDB.faiss")

Total number of embeddings 5654


In [28]:
def load_from_pickle(filename):
    with open(filename, "rb") as file:
        return pickle.load(file)
asd = load_from_pickle("chunks.pickle")

In [29]:
print(asd[1])

page_content='Bij de geactualiseerde Werkwijzer Poortwachter van 1 augustus 2022
In deze werkwijzer is in paragraaf 6.3 de bijzondere arbeidsverhouding uitgewerkt van arbeid met 
een WSW-indicatie en beschut werk vanuit de Participatiewet. Ook zijn op basis van vragen, 
opmerkingen en suggesties teksten aangepast en instructies verduidelijkt. 
Deze tekstwijzigingen zijn gemarkeerd met een blauwe streep in de linker kantlijn.
Addendum COVID-19' metadata={'producer': 'Adobe PDF Library 16.0.7', 'creator': 'Adobe InDesign 17.3 (Windows)', 'creationdate': '2022-08-04T16:16:40+02:00', 'moddate': '2022-08-11T15:31:23+02:00', 'trapped': '/False', 'source': 'Documents/WerkwijzerPoortwachter\\WerkwijzerPoortwachter.pdf', 'total_pages': 42, 'page': 1, 'page_label': '2'}
